# 00 - Exploração dos dados de benefícios concedidos pelo INSS

## Objetivo

Este notebook realiza a exploração inicial do arquivo de benefícios concedidos pelo INSS referente à competência de junho de 2023.

A exploração tem como objetivos:

- compreender o conteúdo e o grão do dataset;
- identificar o volume de registros e colunas;
- analisar os campos disponíveis;
- avaliar a qualidade dos dados;
- identificar valores vazios, zerados ou sem significado de negócio;
- verificar se o dataset permite analisar afastamentos relacionados à saúde mental e às doenças osteomusculares;
- levantar regras que serão aplicadas posteriormente nas camadas Bronze, Silver e Gold.

> Este notebook não cria tabelas das camadas Bronze, Silver ou Gold. O foco desta etapa é exclusivamente conhecer e documentar os dados.

## 1. Importação das funções

Nesta etapa são importadas apenas as funções necessárias para as análises exploratórias.

As funções serão utilizadas para:

- selecionar e filtrar colunas;
- contabilizar registros;
- identificar valores nulos;
- realizar agrupamentos;
- ordenar resultados.

In [0]:
from pyspark.sql.functions import desc

## 2. Definição da origem dos dados

O arquivo foi armazenado em um Volume do Unity Catalog, no schema `bronze`.

Apesar de o arquivo estar fisicamente armazenado no volume da camada Bronze, nenhuma tabela Bronze será criada nesta etapa.

Caminho do arquivo:

`/Volumes/afastamento_inss/bronze/raw/beneficios_concedidos_202306.csv`

In [0]:
path = "/Volumes/afastamento_inss/bronze/raw/beneficios_concedidos_202306.csv"

## 3. Leitura do arquivo CSV

O arquivo é lido com cabeçalho e inferência automática de tipos.

Como o arquivo foi convertido de XLSX para CSV, é necessário validar:

- o delimitador;
- o reconhecimento do cabeçalho;
- a quantidade de colunas;
- os tipos inferidos;
- a preservação de caracteres acentuados.

Nesta exploração, o DataFrame recebe o nome `df`.
`

In [0]:
df = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", ";")
        .csv(path)
)

### Achado

O arquivo foi lido corretamente com 27 colunas.

A validação visual mostrou que os nomes das colunas e os valores acentuados foram preservados. Isso indica que a conversão para CSV e a leitura realizada pelo Spark não deformaram a estrutura observada no arquivo original.

## 4. Volume do dataset

Nesta etapa é calculada a quantidade de linhas e colunas.

Essa medida permite avaliar:

- o tamanho da amostra;
- a viabilidade do processamento;
- a quantidade de atributos disponíveis;
- a abrangência inicial da análise.


In [0]:
print("Linhas:", df.count())
print("Colunas:", len(df.columns))

### Achado

O dataset contém:

- **463.568 registros**
- **27 colunas**

O volume é suficiente para realizar análises descritivas por espécie de benefício, CID, localização geográfica, perfil do beneficiário, ramo de atividade e características previdenciárias.

In [0]:
# Mapeamento explícito de todas as colunas pelo índice real
COL_APS_COD         = df.columns[0]   # 'APS0'
COL_APS_DESC        = df.columns[1]   # 'APS1'
COL_COMPETENCIA     = df.columns[2]   # 'Competência concessão'
COL_ESPECIE_COD     = df.columns[3]   # 'Espécie3'
COL_ESPECIE_DESC    = df.columns[4]   # 'Espécie4'
COL_CID_COD         = df.columns[5]   # 'CID5'
COL_CID_DESC        = df.columns[6]   # 'CID6'
COL_DESPACHO_COD    = df.columns[7]   # 'Despacho7'
COL_DESPACHO_DESC   = df.columns[8]   # 'Despacho8'
COL_DT_NASCIMENTO   = df.columns[9]   # 'Dt Nascimento'
COL_SEXO            = df.columns[10]  # 'Sexo.'
COL_CLIENTELA       = df.columns[11]  # 'Clientela'
COL_MUNICIPIO       = df.columns[12]  # 'Mun Resid'
COL_VINCULO_DEP     = df.columns[13]  # 'Vínculo dependentes'
COL_FORMA_FILIACAO  = df.columns[14]  # 'Forma Filiação'
COL_UF              = df.columns[15]  # 'UF'
COL_QT_SM_RMI      = df.columns[16]  # ' Qt SM RMI '
COL_RAMO_ATIVIDADE  = df.columns[17]  # 'Ramo Atividade'
COL_DT_DCB          = df.columns[18]  # 'Dt DCB'
COL_DT_DDB          = df.columns[19]  # 'Dt DDB'
COL_DT_DIB          = df.columns[20]  # 'Dt DIB'
COL_PAIS_ACORDO     = df.columns[21]  # 'País de Acordo Internacional'
COL_CLASSIFICADOR   = df.columns[22]  # 'Classificador PA'
COL_CNAE_2023       = df.columns[23]  # 'CNAE 2.023'
COL_CNAE_2024       = df.columns[24]  # 'CNAE 2.024'
COL_GRAU_INSTRUCAO  = df.columns[25]  # 'Grau Instrução'
COL_ANOS_CONTRIB    = df.columns[26]  # 'Qt Anos Contribuição'

print("Mapeamento concluído.")
print(f"Total de colunas mapeadas: {len(df.columns)}")

## 5. Avaliação do grão

O grão descreve o que cada linha representa no dataset.

`CID6` e `Espécie4` não constituem o grão isoladamente. Esses campos são atributos utilizados para classificar os registros.

A hipótese inicial é:

> Cada linha representa um registro de benefício concedido pelo INSS.

Entretanto, o dataset explorado não apresentou até este momento um identificador único explícito do benefício. Portanto, a unicidade de cada linha não deve ser afirmada apenas com base em `CID6` ou `Espécie4`.

A avaliação de duplicidade será realizada posteriormente usando o conjunto completo de colunas e combinações de atributos de negócio.

In [0]:
df = spark.range(5)

df.write.mode("overwrite").format("delta").saveAsTable("teste_delta")

display(spark.table("teste_delta"))

### Achado

A comparação entre o total de registros e o total de linhas completamente distintas permite identificar duplicatas exatas.

Uma linha repetida em todas as 27 colunas pode representar:

- uma duplicidade técnica do arquivo; ou
- benefícios distintos sem um identificador individual disponível no dataset.

Por esse motivo, nenhuma remoção automática de duplicatas será realizada durante a exploração. Qualquer regra de deduplicação deverá ser justificada antes da camada Silver.

## 6. Inventário das colunas

O inventário permite identificar:

- nomes com espaços;
- nomes com acentos;
- nomes que começam ou terminam com espaços;
- inconsistências de nomenclatura;
- campos candidatos à padronização na camada Silver.

In [0]:
for c in df.columns:
    print(c)

### Achado

Foram identificados nomes que exigirão padronização, incluindo:

- espaços entre palavras;
- caracteres acentuados;
- pontuação;
- ponto final em `Sexo.`;
- possível espaço inicial e final em ` Qt SM RMI `;
- anos formatados com ponto em `CNAE 2.023` e `CNAE 2.024`.

Na camada Silver, os nomes poderão ser convertidos para um padrão como:

- letras minúsculas;
- ausência de acentos;
- palavras separadas por `_`;
- remoção de espaços no início e no final.

Exemplos:

- `Competência concessão` → `competencia_concessao`
- `Sexo.` → `sexo`
- ` Qt SM RMI ` → `qt_sm_rmi`
- `CNAE 2.023` → `cnae_2023`

## 7. Schema inferido pelo Spark

A inferência automática é usada somente para exploração.

Nesta etapa será verificado se:

- datas foram interpretadas como datas ou textos;
- CNAEs foram tratados como texto ou número;
- códigos CID foram preservados;
- campos quantitativos receberam tipos adequados.

A camada Bronze não deve depender cegamente da inferência automática. O schema definitivo será decidido após a análise.

In [0]:
df.printSchema()

### Interpretação

O schema exibido pelo Spark deve ser comparado com a semântica de cada campo.

Campos como CID, CNAE, município e códigos de classificação podem conter números, mas são códigos de negócio e normalmente devem permanecer como texto. A conversão desses códigos em números pode remover zeros à esquerda ou alterar sua representação.

As datas também devem ser validadas antes da conversão, pois a inferência automática pode mantê-las como texto quando houver formatos inconsistentes.

## 8. Inspeção visual dos registros

A inspeção das primeiras linhas permite entender:

- a forma como códigos e descrições estão combinados;
- os formatos das datas;
- a existência de valores textuais que representam ausência;
- a relação aparente entre espécie, CID e demais atributos.

In [0]:
display(df.limit(20))

### Achado

A inspeção visual indicou que algumas colunas reúnem código e descrição no mesmo valor.

Exemplos observados na exploração incluem:

- CID acompanhado da descrição da doença;
- município contendo identificador e nome;
- espécie contendo a descrição do benefício.

Esses campos poderão ser separados na camada Silver para facilitar filtros, relacionamentos e agregações.


## 9. Contagem de valores nulos

A verificação de nulos identifica apenas valores reconhecidos pelo Spark como `null`.

Essa análise não identifica textos como:

- `Zerados`;
- `Em Branco`;
- string vazia;
- espaços;
- possíveis códigos sentinela.

Portanto, a ausência de nulos não significa ausência de problemas de qualidade.
``

In [0]:
from pyspark.sql.functions import col, count, when

display(
    df.select([
        count(
            when(col(f"`{c}`").isNull(), 1)
        ).alias(c)
        for c in df.columns
    ])
)

### Achado

O resultado apresentou **zero valores nulos em todas as 27 colunas**.

Entretanto, a análise de frequência de `CID6` mostrou valores como `Zerados` e `Em Branco`. Portanto, o arquivo utiliza valores textuais para representar informação ausente ou não aplicável.

Conclusão:

> Não foram encontrados nulos técnicos, mas foram encontrados valores semanticamente vazios.

Esses casos deverão ser tratados separadamente na camada Silver.
``

## 10. Verificação de textos vazios

Além dos valores nulos, será verificada a existência de strings vazias ou compostas apenas por espaços.

In [0]:
from pyspark.sql.functions import trim

colunas_string = [
    campo.name
    for campo in df.schema.fields
    if campo.dataType.simpleString() == "string"
]

contagem_textos_vazios = df.select([
    count(
        when(trim(col(f"`{c}`")) == "", 1)
    ).alias(c)
    for c in colunas_string
])

display(contagem_textos_vazios)

### Achado

A verificação de strings vazias complementa a contagem de nulos.

Os valores encontrados nesta etapa devem ser tratados como problemas de preenchimento, mesmo quando não são reconhecidos tecnicamente como `null`.

## 11. Distribuição dos CIDs

O campo `CID6` é central para classificar os diagnósticos associados aos benefícios.

A análise de frequência ajuda a identificar:

- diagnósticos mais recorrentes;
- valores não informados;
- doenças osteomusculares;
- transtornos mentais;
- necessidade de separar código e descrição.

In [0]:
from pyspark.sql.functions import desc

frequencia_cid = (
    df.groupBy(COL_CID_DESC)
      .count()
      .orderBy(desc("count"))
)

display(frequencia_cid)

### Achados

Os valores mais frequentes observados foram:

- `Zerados`: **242.951 registros**
- `Em Branco`: **16.022 registros**
- `F84.0 Autismo Infantil`: **5.453 registros**
- `M54.5 Dor Lombar Baixa`: **3.462 registros**
- `M51.1 Transtorno de Disco Lombar com comprometimento radicular`: **3.441 registros**
- `D25 Leiomioma do Útero`: **3.117 registros**
- `M51 Outros transtornos de discos intervertebrais`: **2.953 registros**
- `S52.5 Fratura da extremidade distal do rádio`: **2.848 registros**
- `M75.1 Síndrome do manguito rotador`: **2.558 registros**
- `M75 Lesões do ombro`: **2.304 registros**

Os valores `Zerados` e `Em Branco` totalizam **258.973 registros**, aproximadamente **55,87%** das 463.568 linhas.

Essa concentração não deve ser classificada automaticamente como erro. Parte dos benefícios, como aposentadorias, pensões e salário-maternidade, pode não depender de diagnóstico CID para a análise proposta.

A próxima análise deve relacionar os CIDs sem informação à espécie do benefício.

## 12. Distribuição das espécies de benefício

O campo `Espécie4` descreve o tipo de benefício concedido.

Essa análise é necessária porque o dataset não contém apenas afastamentos. A base também inclui aposentadorias, pensões, benefícios assistenciais e outros tipos de concessão.

In [0]:
frequencia_especie = (
    df.groupBy("Espécie4")
      .count()
      .orderBy(desc("count"))
)

display(frequencia_especie)

### Achados

Entre as espécies mais frequentes estão:

- Auxílio Doença Previdenciário: **169.935 registros**
- Aposentadoria por Idade: **73.468 registros**
- Auxílio Salário Maternidade: **52.962 registros**
- Pensão por Morte Previdenciária: **44.952 registros**
- Amparo Social à Pessoa Portadora de Deficiência: **39.333 registros**
- Amparo Social ao Idoso: **33.065 registros**
- Aposentadoria por Tempo de Contribuição: **17.544 registros**
- Auxílio Doença por Acidente do Trabalho: **13.428 registros**
- Aposentadoria por Invalidez Previdenciária: **11.909 registros**

O principal achado desta etapa é:

> O dataset representa benefícios concedidos em geral, e não exclusivamente afastamentos do trabalho.

Consequentemente, a futura camada Silver deverá preservar a base geral e criar uma regra explícita para identificar o subconjunto relevante para a análise de afastamentos.

## 13. Relação entre CID não informado e espécie de benefício

A alta quantidade de valores `Zerados` e `Em Branco` em `CID6` precisa ser interpretada no contexto da espécie do benefício.

A análise abaixo identifica quais espécies concentram esses valores.

In [0]:
cid_sem_informacao = (
    df.filter(
        col("CID6").isin("Zerados", "Em Branco")
    )
    .groupBy("Espécie4", "CID6")
    .count()
    .orderBy(desc("count"))
)

display(cid_sem_informacao)

### Achado — CID sem informação por espécie

O resultado apresentou **29 combinações** de espécie e CID sem informação.

#### Comportamento esperado

A maior parte dos valores `Zerados` está concentrada em benefícios que não dependem de diagnóstico clínico:

| Espécie | CID | Registros |
|---|---|---|
| Aposentadoria por Idade | Zerados | 73.468 |
| Auxílio Salário Maternidade | Zerados | 52.798 |
| Pensão por Morte Previdenciária | Zerados | 44.945 |
| Amparo Social ao Idoso | Zerados | 33.065 |
| Aposentadoria por Tempo de Contribuição | Zerados | 17.544 |
| Amp. Social Pessoa Portadora de Deficiência | Zerados | 10.951 |
| Aposentadoria por Invalidez Previdenciária | Zerados | 6.668 |

Para esses benefícios, a ausência de CID **não representa problema de qualidade**. O campo simplesmente não é aplicável à natureza do benefício.

#### Ponto de atenção — Benefícios de afastamento com CID ausente

Foram encontrados registros de benefícios diretamente relacionados a afastamentos com CID não preenchido:

| Espécie | CID | Registros |
|---|---|---|
| Auxílio Doença Previdenciário | Em Branco | 12.787 |
| Auxílio Acidente | Em Branco | 1.687 |
| Auxílio Acidente Previdenciário | Em Branco | 1.232 |
| Auxílio Doença Previdenciário | Zerados | 430 |
| Auxílio Doença por Acidente do Trabalho | Em Branco | 298 |

O total de registros de afastamento sem CID informado é de aproximadamente **16.434 registros**.

Considerando que o Auxílio Doença Previdenciário possui **169.935 registros** no total, os casos sem CID representam aproximadamente **7,8%** dessa espécie.

#### Conclusão

> A ausência de CID é esperada e justificável para aposentadorias, pensões, amparos e salário-maternidade.
>
> Para os benefícios de afastamento, o CID ausente representa uma limitação real de qualidade que afeta cerca de 7,8% dos registros de Auxílio Doença Previdenciário.

#### Decisão para a camada Silver

Os registros de afastamento com CID `Em Branco` ou `Zerados` **não serão descartados automaticamente**.

Receberão uma classificação explícita:

```text
cid_status = 'nao_informado'

## 14. Distribuição geográfica por UF

A análise por UF verifica a cobertura geográfica e identifica se todas as unidades federativas estão representadas.

In [0]:
frequencia_uf = (
    df.groupBy("UF")
      .count()
      .orderBy(desc("count"))
)

display(frequencia_uf)

### Achado

Foram exibidas **27 categorias de UF**, correspondentes à cobertura observada no arquivo.

Entre os valores visíveis na parte inferior da distribuição estavam:

- Maranhão: 9.420
- Paraíba: 8.376
- Piauí: 7.576
- Espírito Santo: 6.734
- Mato Grosso: 6.314
- Mato Grosso do Sul: 6.309
- Amazonas: 4.621
- Alagoas: 4.002
- Rondônia: 3.475
- Sergipe: 3.449
- Tocantins: 2.198
- Roraima: 1.202
- Amapá: 1.168
- Acre: 1.165

Essa distribuição confirma que o dataset permite análises geográficas. Entretanto, as contagens absolutas não devem ser interpretadas como taxas de afastamento sem um denominador populacional ou de vínculos empregatícios.

# Conclusões da exploração inicial

## Estrutura

- 463.568 registros
- 27 colunas
- arquivo armazenado em Volume do Unity Catalog;
- leitura realizada com Spark;
- nenhuma tabela Bronze criada nesta etapa.

## Grão

A hipótese de trabalho é que cada linha representa um registro de benefício concedido.

Como não foi identificado até o momento um código único explícito do benefício, será necessário avaliar duplicatas e combinações de atributos antes de definir qualquer chave lógica.

## Qualidade

- não foram encontrados valores nulos técnicos;
- existem valores semanticamente vazios, como `Zerados` e `Em Branco`;
- os nomes das colunas precisam ser padronizados;
- algumas colunas combinam código e descrição;
- município aparentemente combina identificador e nome;
- CID e espécie também precisam ser decompostos;
- campos de código devem ser preservados como texto.

## Escopo de negócio

O dataset não contém somente afastamentos. Também estão presentes aposentadorias, pensões, benefícios assistenciais, salário-maternidade e outras espécies.

Para responder à pergunta central do projeto será necessário definir claramente quais espécies representam afastamentos e quais registros devem permanecer apenas na base geral de benefícios.

## Potencial analítico

O dataset permite análises por:

- espécie do benefício;
- CID;
- competência;
- UF e município;
- sexo;
- clientela;
- forma de filiação;
- ramo de atividade;
- CNAE;
- grau de instrução;
- tempo de contribuição.

## Próximas decisões

Antes da camada Bronze deverão ser definidos:

1. schema explícito de leitura;
2. padrão de nomenclatura das colunas;
3. tratamento de código e descrição;
4. política para valores `Zerados` e `Em Branco`;
5. regra de identificação dos afastamentos;
6. tratamento de possíveis duplicatas;
7. metadados técnicos de ingestão.